In [ ]:
from pathlib import Path

import torch
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
import numpy as np

import image_tagger as it

CLIP_MODEL: str = "openai/clip-vit-base-patch32"

In [ ]:
class Clip:
    
    def __init__(self):
        self.model = CLIP_MODEL
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.processor = CLIPProcessor.from_pretrained(self.model)
        self.client = CLIPModel.from_pretrained(self.model).to(self.device)
        self.client.eval()

    def embed_image(self, image: Image.Image) -> np.ndarray:
        """Embed one PIL image as a normalized CLIP vector."""
        return self.embed_images([image])[0]

    def embed_images(self, images: list[Image.Image]) -> np.ndarray:
        """Embed PIL images as normalized CLIP vectors."""
        inputs = self.processor(
            images=[image.convert("RGB") for image in images],
            return_tensors="pt",
        )
        inputs = { name: value.to(self.device) for name, value in inputs.items() }

        with torch.no_grad():
            output = self.client.get_image_features(**inputs)
            features = output.pooler_output
            features = features / features.norm(dim=-1, keepdim=True)

        return features.detach().cpu().to(torch.float64).numpy()

    def embed_text(self, text: str) -> np.ndarray:
        """Embed one text string as a normalized CLIP vector."""
        return self.embed_texts([text])[0]

    def embed_texts(self, texts: list[str]) -> np.ndarray:
        """Embed text strings as normalized CLIP vectors."""
        inputs = self.processor(
            text=texts,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        inputs = { name: value.to(self.device) for name, value in inputs.items() }

        with torch.no_grad():
            output = self.client.get_text_features(**inputs)
            features = output.pooler_output
            features = features / features.norm(dim=-1, keepdim=True)

        return features.detach().cpu().to(torch.float64).numpy()


In [ ]:
clip = Clip()

In [ ]:
images = [
    Image.open(r"C:\Users\oloon\Dropbox\images\uploads\ponytail_littlethunder.jpeg"),
    Image.open(r"C:\Users\oloon\Dropbox\images\uploads\471e22b5d3200df8cde89330f47f216d.jpg"),
]

texts = [
    "A watercolor image of a woman tying her hair up in a ponytail.",
    "The cover of the book *The Thing From Outer Space* by John W Campbell, featuring an orange, three-eyed monster.",
]

In [ ]:
image_vectors = clip.embed_images(images)
text_vectors = clip.embed_texts(texts)
image_vectors @ text_vectors.T

In [ ]:
import random
from io import BytesIO
from PIL import ImageEnhance, ImageFilter, ImageOps


def _apply_random_web_filter(image: Image.Image) -> Image.Image:
    """Apply one plausible web-image color/filter mutation."""
    choice = random.choice([
        "none",
        "grayscale",
        "sepia",
        "warm",
        "cool",
        "low_saturation",
        "high_saturation",
    ])

    # Black-and-white conversion, like old photos or desaturated uploads.
    if choice == "grayscale":
        return ImageOps.grayscale(image).convert("RGB")

    # Sepia tint, like vintage filters and scanned photo effects.
    if choice == "sepia":
        gray = ImageOps.grayscale(image)
        return ImageOps.colorize(gray, black="#24160f", white="#fff0c0")

    # Warmer white balance: slightly more red, slightly less blue.
    if choice == "warm":
        red, green, blue = image.split()
        red = ImageEnhance.Brightness(red).enhance(random.uniform(1.03, 1.12))
        blue = ImageEnhance.Brightness(blue).enhance(random.uniform(0.88, 0.98))
        return Image.merge("RGB", (red, green, blue))

    # Cooler white balance: slightly less red, slightly more blue.
    if choice == "cool":
        red, green, blue = image.split()
        red = ImageEnhance.Brightness(red).enhance(random.uniform(0.88, 0.98))
        blue = ImageEnhance.Brightness(blue).enhance(random.uniform(1.03, 1.12))
        return Image.merge("RGB", (red, green, blue))

    # Washed-out colors, common after filters, compression, or reposting.
    if choice == "low_saturation":
        return ImageEnhance.Color(image).enhance(random.uniform(0.45, 0.85))

    # Punchier colors, common from phone/gallery/social-media filters.
    if choice == "high_saturation":
        return ImageEnhance.Color(image).enhance(random.uniform(1.10, 1.55))

    # No color filter this time.
    return image


def _apply_random_bad_crop_bars(image: Image.Image) -> Image.Image:
    """Add black or white bars that mimic awkward screenshots and bad crops."""
    width, height = image.size
    bar_color = random.choice([(0, 0, 0), (255, 255, 255)])
    bar_mode = random.choice([
        "top",
        "bottom",
        "left",
        "right",
        "horizontal", "horizontal", "horizontal",
        "vertical", "vertical", "vertical",
    ])
    max_bar_width = max(1, round(width * 0.22))
    max_bar_height = max(1, round(height * 0.22))
    horizontal_bar_height = random.randint(1, max_bar_height)
    vertical_bar_width = random.randint(1, max_bar_width)
    mutated = image.copy()

    # Paste bars over image edges to simulate letterboxing, screenshots, and rough crops.
    if bar_mode in {"top", "horizontal"}:
        mutated.paste(bar_color, (0, 0, width, horizontal_bar_height))
    if bar_mode in {"bottom", "horizontal"}:
        mutated.paste(bar_color, (0, height - horizontal_bar_height, width, height))
    if bar_mode in {"left", "vertical"}:
        mutated.paste(bar_color, (0, 0, vertical_bar_width, height))
    if bar_mode in {"right", "vertical"}:
        mutated.paste(bar_color, (width - vertical_bar_width, 0, width, height))

    return mutated


def _apply_random_jpeg_compression(image: Image.Image) -> Image.Image:
    """Round-trip through JPEG to mimic recompressed internet images."""
    buffer = BytesIO()
    image.save(buffer, format="JPEG", quality=random.randint(45, 92))
    buffer.seek(0)
    return Image.open(buffer).convert("RGB")


def randomly_resize_and_crop_image(
    image: Image.Image,
    resize_range: tuple[float, float] = (0.5, 1.1),
    margin_range: tuple[float, float] = (0.0, 0.2),
) -> Image.Image:
    """Copy an image, randomly resize/crop it, then apply plausible web mutations."""
    source = image.convert("RGB").copy()
    width, height = source.size

    # Random rescale: simulates thumbnails, reposts, and downloaded lower-res copies.
    resize_scale = random.uniform(*resize_range)
    resized_size = (
        max(1, round(width * resize_scale)),
        max(1, round(height * resize_scale)),
    )
    mutated = source.resize(resized_size, Image.Resampling.LANCZOS)

    # Four independent random edge crops: simulates screenshots and manual cropping.
    resized_width, resized_height = mutated.size
    left_margin = random.uniform(*margin_range)
    top_margin = random.uniform(*margin_range)
    right_margin = random.uniform(*margin_range)
    bottom_margin = random.uniform(*margin_range)

    # Convert crop fractions into pixel coordinates.
    left = round(resized_width * left_margin)
    top = round(resized_height * top_margin)
    right = round(resized_width * (1.0 - right_margin))
    bottom = round(resized_height * (1.0 - bottom_margin))

    # Guard against pathological random crops that erase the image.
    if right <= left:
        left, right = 0, resized_width
    if bottom <= top:
        top, bottom = 0, resized_height

    # Apply the spatial crop before color/filter mutations.
    mutated = mutated.crop((left, top, right, bottom))

    # Add black or white bars occasionally, like screenshots and poor manual crops.
    if random.random() < 0.20:
        mutated = _apply_random_bad_crop_bars(mutated)

    # Apply one bigger color/filter mutation: grayscale, sepia, warm/cool, etc.
    mutated = _apply_random_web_filter(mutated)

    # Exposure shift: simulates bright/dim uploads and automatic photo corrections.
    if random.random() < 0.75:
        mutated = ImageEnhance.Brightness(mutated).enhance(random.uniform(0.82, 1.18))

    # Contrast shift: simulates edits, scans, and social-media processing.
    if random.random() < 0.75:
        mutated = ImageEnhance.Contrast(mutated).enhance(random.uniform(0.80, 1.25))

    # Sharpness shift: simulates oversharpened thumbnails or softened resized images.
    if random.random() < 0.35:
        mutated = ImageEnhance.Sharpness(mutated).enhance(random.uniform(0.50, 1.80))

    # Mild blur about one third of the time: simulates focus, motion, and resize blur.
    if random.random() < 1 / 3:
        mutated = mutated.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.25, 1.35)))

    # JPEG recompression artifacts: simulates downloaded/reuploaded web images.
    if random.random() < 0.60:
        mutated = _apply_random_jpeg_compression(mutated)

    return mutated

In [ ]:
from IPython.display import display


def show_random_mutation_gallery(
    image: Image.Image,
    k: int = 8,
    canvas_size: int = 1024,
    background: tuple[int, int, int] = (128, 128, 128),
) -> Image.Image:
    """Display a k x k gallery of random image mutations on one square canvas."""
    cell_size = canvas_size // k
    canvas = Image.new("RGB", (canvas_size, canvas_size), background)

    for row in range(k):
        for column in range(k):
            mutation = randomly_resize_and_crop_image(image).convert("RGB")
            mutation.thumbnail(
                (cell_size, cell_size),
                Image.Resampling.LANCZOS,
            )

            x = column * cell_size + (cell_size - mutation.width) // 2
            y = row * cell_size + (cell_size - mutation.height) // 2
            canvas.paste(mutation, (x, y))

    return canvas

In [ ]:
# show_random_mutation_gallery(images[0])

In [ ]:
def show_random_mutation_similarities(
    image: Image.Image,
    k: int = 10,
) -> np.ndarray:
    """Create k random image mutations and show their k x k similarity matrix."""
    mutations = [randomly_resize_and_crop_image(image) for _ in range(k)]
    vectors = clip.embed_images(mutations)
    similarities = vectors @ vectors.T

    return similarities

In [ ]:
# mutation_similarities = show_random_mutation_similarities(images[0], k=8)
# print(np.array2string(mutation_similarities, precision=4, suppress_small=True))

In [ ]:
# distinct_similarities = mutation_similarities[ np.triu_indices_from(mutation_similarities, k=1) ]
# distinct_similarities.mean(), distinct_similarities.min()

In [ ]:
image_paths = it.find_images(r"C:\Users\oloon\Dropbox\images\art\jacek_yerka")
len(image_paths)

In [ ]:
def embed_image_paths(
    image_paths: list[Path],
    batch_size: int = 32,
) -> np.ndarray:
    vectors = []

    for start in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[start:start + batch_size]
        batch_images = []

        for path in batch_paths:
            with Image.open(path) as image:
                batch_images.append(image.convert("RGB").copy())

        vectors.append(clip.embed_images(batch_images))
        print(f"{min(start + batch_size, len(image_paths))}/{len(image_paths)}")

    return np.vstack(vectors)

In [ ]:
embedding_matrix = embed_image_paths(image_paths, batch_size=4)
similarity_matrix = embedding_matrix @ embedding_matrix.T
similarity_matrix.shape

threshold = 0.90
matches = [ (int(i), int(j)) for i, j in zip(*np.where(np.triu(similarity_matrix > threshold, k=1))) ]

print(len(matches))

for score, i, j, left_path, right_path in sorted([
        (float(similarity_matrix[i, j]), i, j, Path(image_paths[i]).name, Path(image_paths[j]).name)
        for i, j in matches
    ], reverse=True):
    print(f"{left_path!r} == {right_path!r} ({score*100:0.0f}%)")

In [ ]:
image_paths = it.find_images(r"C:\Users\oloon\Dropbox\images")

image_paths = [ path for path in image_paths if "ai" not in path.parts ]
len(image_paths)

In [ ]:
embedding_matrix = embed_image_paths(image_paths, batch_size=64)
similarity_matrix = embedding_matrix @ embedding_matrix.T
similarity_matrix.shape

In [ ]:

threshold = 0.95
matches = [ (int(i), int(j)) for i, j in zip(*np.where(np.triu(similarity_matrix > threshold, k=1))) ]

print(len(matches))

for score, i, j, left_path, right_path in sorted([
        (float(similarity_matrix[i, j]), i, j, Path(image_paths[i]).name, Path(image_paths[j]).name)
        for i, j in matches
    ], reverse=True):
    print(f"{left_path!r} == {right_path!r} ({score*100:0.0f}%)")

In [ ]:
import base64
import html
import json
from pydantic import BaseModel
from IPython.display import HTML, display


class SameImageJudgement(BaseModel):
    thinking: str
    same_image: bool


SAME_IMAGE_PROMPT: str = """
Are these the exact same image except for minor cropping, resizing, and filtering?

Return true only when the images appear to come from the same original picture.
Treat changes in crop, scale, compression, color filtering, exposure, contrast,
letterboxing, and small border bars as minor. Return false for merely similar
subjects, compositions, styles, or scenes.
""".strip()


def _image_to_data_url(
    image_path: str | Path,
    max_size: tuple[int, int] = (320, 320),
) -> str:
    """Convert an image path to a base64 JPEG data URL for notebook HTML."""
    image = Image.open(image_path).convert("RGB")
    image.thumbnail(max_size, Image.Resampling.LANCZOS)

    buffer = BytesIO()
    image.save(buffer, format="JPEG", quality=88)
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/jpeg;base64,{encoded}"


def _image_to_vision_base64(
    image_path: str | Path,
    max_dimension: int = 768,
) -> str:
    """Resize and encode one image for a vision model request."""
    image = it.resize_image_to_fit(image_path, max_dimension=max_dimension)
    return it.base64_encode_image(image)


def _display_image_matches(
    matches: list[tuple[float, int, int]],
    image_paths: list[str | Path],
    judgements: dict[tuple[int, int], SameImageJudgement] | None = None,
    max_size: tuple[int, int] = (320, 320),
) -> None:
    """Display ranked image match rows in notebook HTML."""
    rows = []
    for rank, (score, i, j) in enumerate(matches, start=1):
        left_path = Path(image_paths[i])
        right_path = Path(image_paths[j])
        left_name = html.escape(left_path.name)
        right_name = html.escape(right_path.name)
        left_url = _image_to_data_url(left_path, max_size=max_size)
        right_url = _image_to_data_url(right_path, max_size=max_size)
        judgement = judgements.get((i, j)) if judgements is not None else None
        reason = html.escape(judgement.thinking) if judgement is not None else ""
        reason_html = f'<p class="match-reason">{reason}</p>' if reason else ""

        rows.append(
            f"""
            <div class="match-row">
                <div class="match-meta">
                    <strong>#{rank}</strong>
                    <span>{score * 100:0.2f}%</span>
                    <code>({i}, {j})</code>
                </div>
                <figure>
                    <img src="{left_url}" alt="{left_name}">
                    <figcaption>{left_name}</figcaption>
                </figure>
                <figure>
                    <img src="{right_url}" alt="{right_name}">
                    <figcaption>{right_name}</figcaption>
                </figure>
                {reason_html}
            </div>
            """,
        )

    display(HTML(
        f"""
        <style>
            .match-list {{
                display: grid;
                gap: 14px;
                max-width: 980px;
            }}
            .match-row {{
                align-items: start;
                border: 1px solid #d0d7de;
                border-radius: 6px;
                display: grid;
                gap: 12px;
                grid-template-columns: 120px minmax(0, 1fr) minmax(0, 1fr);
                padding: 12px;
            }}
            .match-meta {{
                display: grid;
                gap: 6px;
                font: 13px/1.35 sans-serif;
            }}
            .match-meta strong {{
                font-size: 16px;
            }}
            .match-meta code {{
                white-space: nowrap;
            }}
            .match-row figure {{
                margin: 0;
                min-width: 0;
            }}
            .match-row img {{
                background: #f6f8fa;
                display: block;
                height: auto;
                max-height: {max_size[1]}px;
                max-width: 100%;
                object-fit: contain;
            }}
            .match-row figcaption {{
                font: 12px/1.35 sans-serif;
                margin-top: 6px;
                overflow-wrap: anywhere;
            }}
            .match-reason {{
                font: 12px/1.45 sans-serif;
                grid-column: 2 / 4;
                margin: 0;
            }}
        </style>
        <div class="match-list">
            {''.join(rows)}
        </div>
        """,
    ))


def top_image_matches(
    similarity_matrix: np.ndarray,
    n: int = 10,
    threshold: float | None = None,
) -> list[tuple[float, int, int]]:
    """Return the top N upper-triangle matches as score, i, j tuples."""
    upper_i, upper_j = np.triu_indices_from(similarity_matrix, k=1)
    scores = similarity_matrix[upper_i, upper_j]

    if threshold is not None:
        keep = scores > threshold
        upper_i = upper_i[keep]
        upper_j = upper_j[keep]
        scores = scores[keep]

    order = np.argsort(scores)[::-1][:n]
    return [
        (float(scores[index]), int(upper_i[index]), int(upper_j[index]))
        for index in order
    ]


def show_top_image_matches(
    similarity_matrix: np.ndarray,
    image_paths: list[str | Path],
    n: int = 10,
    threshold: float | None = None,
    max_size: tuple[int, int] = (320, 320),
) -> list[tuple[float, int, int]]:
    """Show the top N upper-triangle image matches side-by-side in notebook HTML."""
    matches = top_image_matches(similarity_matrix, n=n, threshold=threshold)
    _display_image_matches(matches, image_paths, max_size=max_size)
    return matches


def judge_same_image_match(
    left_path: str | Path,
    right_path: str | Path,
    client_adapter: it.VisionModelClientAdapter | None = None,
    max_dimension: int = 768,
) -> SameImageJudgement:
    """Ask a vision model whether two files are the same underlying image."""
    if client_adapter is None:
        client_adapter = it.get_vision_model_client_adapter(it.VisionModelProvider.OPENAI)

    response = client_adapter.vision_task(
        [
            _image_to_vision_base64(left_path, max_dimension=max_dimension),
            _image_to_vision_base64(right_path, max_dimension=max_dimension),
        ],
        SAME_IMAGE_PROMPT,
        SameImageJudgement,
    )
    return SameImageJudgement.model_validate_json(response.content)


def show_llm_filtered_image_matches(
    similarity_matrix: np.ndarray,
    image_paths: list[str | Path],
    n: int = 10,
    threshold: float = 0.90,
    provider: it.VisionModelProvider = it.VisionModelProvider.OPENAI,
    max_size: tuple[int, int] = (320, 320),
    max_dimension: int = 768,
) -> list[tuple[float, int, int]]:
    """Use CLIP for candidates, then display matches accepted by a vision LLM."""
    client_adapter = it.get_vision_model_client_adapter(provider)
    candidates = top_image_matches(similarity_matrix, n=n, threshold=threshold)
    surviving_matches: list[tuple[float, int, int]] = []
    judgements: dict[tuple[int, int], SameImageJudgement] = {}

    try:
        for score, i, j in candidates:
            judgement = judge_same_image_match(
                image_paths[i],
                image_paths[j],
                client_adapter=client_adapter,
                max_dimension=max_dimension,
            )
            print(image_paths[i], "==",image_paths[j], "?")
            print(judgement)
            judgements[(i, j)] = judgement
            if judgement.same_image:
                surviving_matches.append((score, i, j))
    finally:
        client_adapter.cleanup()

    _display_image_matches(
        surviving_matches,
        image_paths,
        judgements=judgements,
        max_size=max_size,
    )
    return surviving_matches


In [ ]:
show_llm_filtered_image_matches(similarity_matrix, image_paths, threshold=0.90, n=100)